brokered by (categorically encoded agency/broker)

status (Housing status - a. ready for sale or b. ready to build)

price (Housing price, it is either the current listing price or recently sold price if the house is sold recently)

bed (# of beds)

bath (# of bathrooms)

acre_lot (Property / Land size in acres)

street (categorically encoded street address)

city (city name)

state (state name)

zip_code (postal code of the area)

house_size (house area/size/living space in square feet)

prev_sold_date (Previously sold date)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('realtor-data.zip.csv')
print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
df.head()

Dataset shape: 2,226,382 rows, 12 columns


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


---
## 3. Data Cleaning & Preprocessing

In [12]:
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(2)

=== Dataset Info ===
<class 'pandas.DataFrame'>
Index: 2223239 entries, 0 to 2226381
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   brokered_by         object 
 1   status              str    
 2   price               float64
 3   bed                 float64
 4   bath                float64
 5   acre_lot            float64
 6   street              object 
 7   city                str    
 8   state               str    
 9   zip_code            float64
 10  house_size          float64
 11  prev_sold_date      str    
 12  is_previously_sold  int64  
dtypes: float64(6), int64(1), object(2), str(4)
memory usage: 237.5+ MB

=== Statistical Summary ===


,price,bed,bath,acre_lot,zip_code,house_size,is_previously_sold
count,2.223239e+06,2223239.00,2223239.00,2223239.00,2223239.00,2.223239e+06,2223239.00
mean,5.232252e+05,3.22,2.46,12.99,52185.09,2.490520e+03,0.67
std,1.582023e+06,1.40,1.47,704.60,28961.54,6.978885e+05,0.47
min,0.000000e+00,1.00,1.00,0.00,0.00,4.000000e+00,0.00
25%,1.650000e+05,3.00,2.00,0.15,29615.00,1.400000e+03,0.00
50%,3.250000e+05,3.00,2.00,0.25,48386.00,1.760000e+03,1.00
75%,5.500000e+05,4.00,3.00,0.75,78070.00,2.240000e+03,1.00
max,1.000000e+09,473.00,830.00,100000.00,99999.00,1.040400e+09,1.00


## Missing values

In [3]:
# Missing values
missing_counts = df.isnull().sum()

missing_percentage = (missing_counts / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_percentage
})

missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("Missing Values Percentage per Column:")
print(missing_info)

Missing Values Percentage per Column:
                Missing Count  Missing Percentage
prev_sold_date         734297           32.981627
house_size             568484           25.533983
bath                   511771           22.986666
bed                    481317           21.618797
acre_lot               325589           14.624130
street                  10866            0.488056
brokered_by              4533            0.203604
price                    1541            0.069215
city                     1407            0.063197
zip_code                  299            0.013430
state                       8            0.000359


In [4]:
# Calculate the overall median of 'house_size' column
overall_median = df['house_size'].median()

#fill null values in 'house_size' column with the median of the respective city
df['house_size'] = df.groupby('city')['house_size'].transform(lambda x: x.fillna(x.median()))

# fill null values in 'house_size' column with the overall median if there are still any nulls left
df['house_size'] = df['house_size'].fillna(overall_median)

In [5]:
df['size_group'] = pd.qcut(df['house_size'], q=5, labels=['XS', 'S', 'M', 'L', 'XL'])

df['bed'] = df.groupby('size_group')['bed'].transform(lambda x: x.fillna(x.median()))
df['bath'] = df.groupby('size_group')['bath'].transform(lambda x: x.fillna(x.median()))

df = df.drop(columns=['size_group'])

In [8]:
# 1. first get the overall median of 'acre_lot' column
overall_median_acre = df['acre_lot'].median()

# 2. fill null values in 'acre_lot' column with the median of the respective city
df['acre_lot'] = df.groupby('city')['acre_lot'].transform(lambda x: x.fillna(x.median()))

# 3. if any null values are still left in 'acre_lot' column, fill them with the overall median
df['acre_lot'] = df['acre_lot'].fillna(overall_median_acre)

In [9]:
# fill null values in 'street' and 'brokered_by' columns with 'Unknown'
df['street'] = df['street'].fillna('Unknown')
df['brokered_by'] = df['brokered_by'].fillna('Unknown')

In [10]:
# drop rows with null values in 'price', 'city', 'zip_code', and 'state' columns
df = df.dropna(subset=['price', 'city', 'zip_code', 'state'])

Insight: Dropping Rows with Missing Target and Low-Percentage Features

Target Variable (price): Imputing missing values in the target variable can introduce artificial bias and force the model to learn incorrect patterns. Dropping these rows ensures model integrity.

Low-Percentage Features (city, zip_code, state): Since the missing data in these columns is negligible (less than 0.1%), dropping them eliminates unnecessary guesswork without causing any significant data loss.

In [14]:
# duplicate rows
print(f'Duplicate rows: {df.duplicated().sum():,}')
df.drop_duplicates(inplace=True)
print(f'Duplicate rows after dropping: {df.duplicated().sum():,}')
print(f'Cleaned dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

Duplicate rows: 45
Duplicate rows after dropping: 0
Cleaned dataset shape: 2,223,194 rows x 13 columns


In [15]:
df.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,is_previously_sold
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN,0
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN,0
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN,0
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN,0
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,1550.0,NaN,0


In [20]:
df['status'].value_counts()

status
for_sale          1386839
sold               811643
ready_to_build      24712
Name: count, dtype: int64

# Feature Engineering 

In [7]:
df['is_previously_sold'] = df['prev_sold_date'].notna().astype(int)

Insight : We kept NaN values as they are: Nearly 33% of the data in the prev_sold_date column is missing. Converting these missing values into a dummy string like "None" or a fake date would change the data type of the entire column to an object/string. This would break future datetime operations and calculations. Keeping them as NaN preserves the correct data type and allows modern machine learning algorithms (like XGBoost or LightGBM) to handle them natively without creating any artificial bias.

In [21]:
# --- Conversion Step ---
# Convert the 'prev_sold_date' column to datetime
df['prev_sold_date'] = pd.to_datetime(df['prev_sold_date'], errors='coerce')

print("\nDataFrame info after conversion:")
df.info()
print("\n'prev_sold_date' column after conversion:")
print(df['prev_sold_date'])


DataFrame info after conversion:
<class 'pandas.DataFrame'>
Index: 2223194 entries, 0 to 2226381
Data columns (total 13 columns):
 #   Column              Dtype         
---  ------              -----         
 0   brokered_by         object        
 1   status              str           
 2   price               float64       
 3   bed                 float64       
 4   bath                float64       
 5   acre_lot            float64       
 6   street              object        
 7   city                str           
 8   state               str           
 9   zip_code            float64       
 10  house_size          float64       
 11  prev_sold_date      datetime64[us]
 12  is_previously_sold  int64         
dtypes: datetime64[us](1), float64(6), int64(1), object(2), str(3)
memory usage: 237.5+ MB

'prev_sold_date' column after conversion:
0                NaT
1                NaT
2                NaT
3                NaT
4                NaT
             ...    
2226377  

In [28]:
df[df['prev_sold_date'] >= pd.Timestamp('2026-06-11')]

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,is_previously_sold
1474557,89918.0,sold,325000.0,2.0,2.0,0.15,598435.0,Long Beach,New York,11561.0,1012.0,3019-04-02,1


In [29]:
# drop rows with 'prev_sold_date' greater than or equal to '2026-06-11'
df = df[df['prev_sold_date'] != '3019-04-02']